# ReceiptLens — Theme 3B Hack Day Runbook

**Goal:** Fine-tune **Gemma 4 E2B Vision** on **ICDAR 2019 SROIE** receipts with Unsloth LoRA, and compare the original vs adapted model on *the same held-out receipt images*. Build a Gradio demo and export a reproducible report.

**Run each cell from top to bottom.** The first download is significant, and a T4 GPU allocation is not guaranteed. This is a **prebuilt project scaffold**, not evidence of a completed training run. **Never put private API keys or customer receipts in public links.**

**Before you begin:** Google Colab → Runtime → Change runtime type → **T4 GPU**. Make sure GPU is connected. Keep the attached ZIP (`ReceiptLens_HackDay_Complete.zip`) available on your machine for step 2.

Official starting points:
- [Organizer's fine-tuning problem statement](https://github.com/mithun-srinivas/HacktoberFest-HackDay/blob/main/problem-statements/03_fine_tuning.md)
- [Organizer's original notebook](https://github.com/mithun-srinivas/HacktoberFest-HackDay/blob/main/templates/03_fine_tuning_starter_kit.ipynb)
- [Official Unsloth Gemma 4 E2B Vision notebook](https://github.com/unslothai/notebooks/blob/main/nb/Gemma4_(E2B)-Vision.ipynb)

**Methodology:** 160 training examples / 20 validation examples from original training split; 30 **untouched** examples from official test split. Start with a quick smoke run, then adjust as allowed by Colab GPU. The final-test sample is used for reporting only, never to tune hyperparameters.

## 1 — Check GPU
**Expected:** `True` and an NVIDIA GPU such as Tesla T4. If you see False, stop and choose a GPU runtime.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime > Change runtime type > T4 GPU")
print("GPU:", torch.cuda.get_device_name(0))
print("Total VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

## 2 — Upload the complete project ZIP
Upload `ReceiptLens_HackDay_Complete.zip` (not just the notebook). This makes all `src/receiptlens/*.py` modules and tests available in this session. The archive contains **no trained model weights or dataset images**.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, os, sys
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith(".zip")]
if not archives:
    raise FileNotFoundError("Upload ReceiptLens_HackDay_Complete.zip")
workspace = Path("/content/receiptlens_workspace")
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as z:
    # All entries come from the trusted project ZIP you downloaded.
    z.extractall(workspace)
project_dir = workspace / "ReceiptLens_HackDay"
if not (project_dir / "src/receiptlens/vision.py").exists():
    raise RuntimeError("Expected project folder inside ZIP. Check uploaded file.")
os.chdir(project_dir)
sys.path.insert(0, str(project_dir / "src"))
print("Project:", project_dir)
print("Python files:", len(list((project_dir/'src').rglob('*.py'))))

## 3 — Install GPU training dependencies (same approach as organizer)
This cell intentionally follows the organizer's pinned installation sequence. **If it fails**, compare it with the most recent organizer / official Unsloth Colab notebook; don't randomly upgrade libraries mid-training. Remove `%%capture` while debugging. A runtime restart may be necessary after installs.

In [ ]:
%%capture
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
!pip install "gradio>=6,<7" pandas matplotlib pillow pytest

## 4 — Lightweight tests and configuration
These tests validate parsing, schema, preprocessing and evaluation logic only. **They do not test GPU training.**

In [ ]:
!python -m pytest -q tests

In [ ]:
from receiptlens.config import ExperimentConfig
from receiptlens.prompts import FIELDS, RECEIPT_INSTRUCTION

CFG = ExperimentConfig(
    train_count=160,
    validation_count=20,
    test_count=30,
    max_image_edge=960,
    lora_rank=8,
    lora_alpha=8,
    train_steps=40,
)
CFG.validate()
CFG.save("outputs/experiment_config.json")
print(CFG)
print("Instruction:", RECEIPT_INSTRUCTION)

## 5 — Download SROIE and inspect a real annotated receipt
This can download roughly hundreds of MB and will take some time. **Do not feed `entities`, `words`, or `bboxes` into the model.** They are annotation data. The model sees only the receipt image and fixed instruction.

In [ ]:
from receiptlens.data import load_sroie, select_splits
from IPython.display import display
raw = load_sroie(CFG.dataset_id)
print("Available dataset splits:", {k: len(raw[k]) for k in raw})
example = raw["train"][0]
display(example["image"])
print("Ground truth:", example["entities"])
assert all(field in example["entities"] for field in FIELDS)

## 6 — Make reproducible splits
**Training and validation are separate portions of the original training split.** The official test split stays untouched and is not used for model selection.

In [ ]:
splits = select_splits(raw, CFG.train_count, CFG.validation_count, CFG.test_count, CFG.seed)
print({key: len(value) for key,value in splits.items()})
train_ds = splits["train"]
validation_ds = splits["validation"]
test_ds = splits["test"]

## 7 — Load original Gemma 4 E2B vision model
The model is open-weight and downloaded on first call. **This cell requires an NVIDIA CUDA GPU.** Models and Unsloth versions can change; refer to official Vision notebook if an import breaks.

In [ ]:
from receiptlens.vision import load_model, predict_receipt, add_lora, save_adapter
model, processor = load_model(CFG.model_id)
print("Gemma vision model loaded")

## 8 — Original-model single-image smoke test
**Pass condition:** the model produces text from an image without a CUDA or chat-template error. Output may be inaccurate; that's precisely why we're evaluating it.

In [ ]:
val_receipt = validation_ds[0]
base_smoke = predict_receipt(model, processor, val_receipt["image"],
    max_edge=CFG.max_image_edge, max_new_tokens=CFG.max_new_tokens)
print("BASELINE OUTPUT:", base_smoke)
print("EXPECTED:", val_receipt["entities"])

## 9 — Baseline validation
Measure initial accuracy on **validation** only. You can adjust settings based on these results if you do the same for both models, but make the planned comparison protocol explicit in your writeup.

In [ ]:
from receiptlens.metrics import evaluate_samples, compare_results
from receiptlens.reports import save_report, plot_comparison
from functools import partial
base_predict = partial(predict_receipt, model, processor,
                       max_edge=CFG.max_image_edge, max_new_tokens=CFG.max_new_tokens)
base_validation = evaluate_samples(validation_ds, base_predict,
                                   label="plain_validation", max_count=5)
display(base_validation[["index","valid_schema","company","date","address","total","all_four","error"]])

## 10 — Cache the original model's **final-test** predictions *once, before fine-tuning*
Keep this set locked. The cached results make the final comparison reproducible even if the adapter cannot be switched off live. **Do not look at these results to adjust hyperparameters.** In a time crunch reduce `test_count` only before looking at the results, and report actual N.

**Important:** if inference costs too much GPU time, start with `test_count=10` while debugging and run a single fresh, locked final comparison later.

In [ ]:
plain_test_df = evaluate_samples(test_ds, base_predict, label="plain_model")
plain_test_df.to_csv("outputs/baseline_predictions.csv", index=False)
print("Saved baseline_predictions.csv (exact images and prompt fixed)")

## 11 — Attach LoRA adapters
Keep the base model weights frozen; train LoRA matrices in the language and vision layers. If CUDA OOM occurs, restart and lower image resolution (e.g. 768), then recompute the **entire** baseline fairly with the new preprocessing.

In [ ]:
model = add_lora(model, rank=CFG.lora_rank, alpha=CFG.lora_alpha)
print("LoRA adapters initialized; rank", CFG.lora_rank)

## 12 — Create the vision fine-tuning trainer
The **vision data collator is mandatory** here. This is not the organizer's text-only ticket-triage training loop.

In [ ]:
from receiptlens.train import make_trainer
trainer = make_trainer(model, processor, train_ds, CFG)
print("Trainer configured:", len(train_ds), "training receipt images")

## 13 — Train
Before your main run: optional smoke training with `train_steps=10` and `train_count=60` in CFG, then restart model for actual run. Do **not** keep running a smoke fine-tune and call the next one a fresh baseline.

A decreasing training loss does not prove better extraction. Your held-out evaluation does.

In [ ]:
train_result = trainer.train()
model = trainer.model
print("Training metrics:", train_result.metrics)

## 14 — Save LoRA adapters immediately
Output folder contains adapters and processor config, **not a standalone copy of the full base model**. Copy this folder to Google Drive or download the ZIP below before the Colab session ends.

In [ ]:
adapter_path = save_adapter(model, processor, "outputs/receipt_gemma4_lora")
print("Saved LoRA adapter:", adapter_path)

## 15 — Evaluate the fine-tuned model on the same final-test receipts
Identical instruction, image size, generation parameters, annotation split, and scoring.

In [ ]:
tuned_predict = partial(predict_receipt, model, processor,
                        max_edge=CFG.max_image_edge, max_new_tokens=CFG.max_new_tokens)
finetuned_test_df = evaluate_samples(test_ds, tuned_predict, label="fine_tuned_model")
comparison = save_report(plain_test_df, finetuned_test_df, CFG, "outputs")
display(comparison)
print("Chart:", plot_comparison(comparison))

## 16 — Investigate failure cases
Don't hide negative results. Inspect incorrect predictions by field, invalid JSON and examples where adaptation did not help. Note that matching scores may vary due to punctuation and date/amount normalization.

In [ ]:
from IPython.display import display
cols = ["index","company","date","address","total","all_four","prediction","ground_truth"]
display(finetuned_test_df.loc[finetuned_test_df["all_four"] == 0, cols].head(5))

## 17 — Launch a receipt-upload demo
The demo generates LoRA-fine-tuned predictions for uploaded images. It only displays a **live plain-model** prediction if `disable_adapter()` works on this model; otherwise it clearly tells users to inspect the baseline evaluation CSV. No fake predictions. The `gradio.live` URL is a temporary public link while the runtime stays connected.

In [ ]:
from receiptlens.demo import make_demo
import gradio as gr
gr.close_all()
app = make_demo(model, processor, CFG)
app.launch(share=True, debug=False)

## 18 — Export results and adapters
Save your notebook separately in Google Drive (File → Save a copy in Drive). The ZIP below includes comparison, configuration, saved adapters and predictions. Download it, then archive or link from a project README. Do not commit raw receipt images or huge model files to GitHub.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/receiptlens_results", "zip", root_dir=project_dir / "outputs")
print("Result archive:", archive)
files.download(archive)

## 19 — Optional: Google Drive backup
If you have Drive access, also save the archive there; otherwise download locally. This step requires explicit Drive authorization and should not be run for a shared account without permission.

In [ ]:
# OPTIONAL. Remove the leading # only if you want Google Drive backup.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy2('/content/receiptlens_results.zip', '/content/drive/MyDrive/receiptlens_results.zip')

## 20 — Submission and demo checklist

- [ ] Explain problem and scope (receipt → company/date/address/total JSON).
- [ ] Describe **SROIE** source, copyright/license, actual split sizes and limitations.
- [ ] Show baseline prediction and baseline metric values on fixed held-out set.
- [ ] Document LoRA rank, image size, learning rate, training steps, GPU and seed.
- [ ] Show **measured** fine-tuned score comparison and one failure case.
- [ ] Demonstrate upload of one unseen receipt image in Gradio.
- [ ] Save adapters, Colab notebook, `comparison.csv`, and a README with reproduction steps.
- [ ] State limitations and avoid claims about accuracy on Indian receipts without testing.

**Backup strategy:** If training fails, retain the baseline notebook, dataset inspection, tested evaluation code, and log the exact blocker transparently. Don't fabricate the fine-tuned model or scores.

**Disclaimer:** This scaffold was built in advance; training, live inference, and published metrics should be produced during your actual experiment.